# Computer-Aided Drug Discovery: LSD1 (KDM1A) Inhibitors

**Target:** Lysine-specific histone demethylase 1A
**ChEMBL ID:** CHEMBL6136 | **UniProt:** O60341
**Structure:** PDB 6TUY (LSD1/CoREST + FAD + reversible quinazoline MC4106, 2.60 A)
**Disease context:** Acute myeloid leukaemia, small-cell lung cancer

LSD1 is a FAD-dependent histone demethylase that removes methyl marks from
histone H3 lysine 4. It is an epigenetic "eraser" and a validated oncology
target.

---

## Pipeline

| Phase | Content |
|---|---|
| 1 | Data acquisition from ChEMBL |
| 2 | Molecular curation and activity classification |
| 3 | Physicochemical descriptor analysis |
| 4 | Fragment and scaffold analysis |
| 5 | Similarity screening (ZINC + drug repurposing library) |
| 6 | Molecular docking, enrichment, interaction analysis |

## How to run

`Runtime` -> `Run all`, then leave it.

| Stage | Approximate time |
|---|---|
| Phases 1-4 | ~10 min (mostly the ChEMBL download) |
| Phase 5 | ~15 min (library download + screening) |
| Phase 6 install | ~5 min |
| Phase 6 docking | **1-2 hours** |

Docking results are written to disk after every ligand, so if the session
disconnects you can simply re-run that cell and it resumes.

**Set `QUICK = True` in the configuration cell below to test the whole
pipeline in a few minutes before committing to the full run.**

# Phase 1 - Data Acquisition

## 1.1 Environment setup

Installs dependencies and sets the working directory. Re-run at the start of
every Colab session; installed packages do not persist between sessions.

In [ ]:
# ======================= CONFIGURATION =======================
# Set QUICK = True for a fast end-to-end test of the pipeline.
# Set QUICK = False for the full run used in the final report.

QUICK = False

if QUICK:
    N_ACT, N_INA   = 20, 20      # known actives / inactives to dock
    N_ZINC, N_DRUG = 20, 20      # library hits to dock
    EXHAUSTIVENESS = 4           # Vina search effort
else:
    N_ACT, N_INA   = 100, 100
    N_ZINC, N_DRUG = 100, 100
    EXHAUSTIVENESS = 8

print("QUICK mode:", QUICK)
print(f"ligands to dock ~ {N_ACT + N_INA + N_ZINC + N_DRUG}")
print("exhaustiveness  :", EXHAUSTIVENESS)
# =============================================================

In [ ]:
!pip install -q rdkit chembl_webresource_client

import os

# Try to persist results to Google Drive; fall back to session storage.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = "/content/drive/MyDrive/CADD_LSD1"
except Exception as e:
    print("Drive unavailable (", type(e).__name__, ") - using session storage.")
    print("IMPORTANT: download your output files before closing this tab.")
    PROJECT = "/content/CADD_LSD1"

os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
print("Working directory:", os.getcwd())

## 1.2 Download bioactivity data

All IC50 records for CHEMBL6136 are retrieved from the ChEMBL web services
API. Records are fetched 1000 at a time; the guard prevents re-downloading
when the file already exists.

In [ ]:
import requests
import pandas as pd
import numpy as np

RAW = "LSD1_IC50_raw.csv"

if not os.path.exists(RAW):
    BASE = "https://www.ebi.ac.uk/chembl/api/data/activity"
    params = {
        "target_chembl_id": "CHEMBL6136",
        "standard_type": "IC50",
        "format": "json",
        "limit": 1000,
        "offset": 0,
    }
    rows = []
    while True:
        r = requests.get(BASE, params=params, timeout=120).json()
        rows.extend(r["activities"])
        print("fetched:", len(rows))
        if not r["page_meta"]["next"]:
            break
        params["offset"] += 1000

    keep = ["molecule_chembl_id", "canonical_smiles", "standard_value",
            "standard_units", "standard_relation", "pchembl_value",
            "assay_chembl_id", "assay_description", "assay_type",
            "document_chembl_id"]
    pd.DataFrame(rows)[keep].to_csv(RAW, index=False)
    print("Saved", RAW)
else:
    print("Using existing", RAW)

## 1.3 Inspect the raw dataset

Before any filtering, establish what the data contains: units, measurement
relations, missing values and compound count.

**Why this matters.** IC50 values recorded in different units cannot be
log-transformed together. Checking first prevents a silent error that would
propagate through the entire analysis.

In [ ]:
df = pd.read_csv(RAW)

print("RAW SHAPE:", df.shape)
print("\n--- UNITS ---")
print(df["standard_units"].value_counts(dropna=False))
print("\n--- RELATIONS ---")
print(df["standard_relation"].value_counts(dropna=False))
print("\n--- MISSING ---")
print(df[["canonical_smiles", "standard_value", "pchembl_value"]].isna().sum())
print("\nUnique compounds:", df["molecule_chembl_id"].nunique())

## 1.4 Filter to analysable records

| Filter | Reason |
|---|---|
| `units == nM` | Mixed units cannot be pooled |
| `relation == '='` | `<` and `>` are censored limits, not measurements |
| value and SMILES present | Required downstream |
| `value > 0` | Zero or negative values break the logarithm |

A funnel is printed so every discarded row is accounted for. Censored
records (`<`, `>`) make up a substantial fraction of the raw data; their
removal is deliberate and is reported in the methods.

In [ ]:
d = df.copy()
n = len(d)
print(f"start                 {n:>6}")

d = d[d["standard_units"] == "nM"]
print(f"units == nM           {len(d):>6}   (-{n-len(d)})"); n = len(d)

d = d[d["standard_relation"] == "="]
print(f"relation == '='       {len(d):>6}   (-{n-len(d)})"); n = len(d)

d = d.dropna(subset=["standard_value", "canonical_smiles"])
print(f"has value + SMILES    {len(d):>6}   (-{n-len(d)})"); n = len(d)

d = d[d["standard_value"] > 0]
print(f"value > 0             {len(d):>6}   (-{n-len(d)})")

## 1.5 Convert IC50 to pIC50

$$pIC_{50} = -\log_{10}(IC_{50}\ \text{in M}) = 9 - \log_{10}(IC_{50}\ \text{in nM})$$

Higher pIC50 means greater potency. The log scale makes the distribution
approximately symmetric and suitable for statistical testing.

**Validation.** Computed values are compared against ChEMBL's own
`pchembl_value` column - an independent check of both the formula and the
unit filter. Agreement to within rounding confirms the conversion.

In [ ]:
d = d.copy()
d["pIC50"] = 9 - np.log10(d["standard_value"])

chk = d.dropna(subset=["pchembl_value"])
diff = (chk["pIC50"] - chk["pchembl_value"]).abs()

print("Rows cross-checked:", len(chk))
print("Max difference:", round(diff.max(), 4))
print("Rows differing by >0.01:", (diff > 0.01).sum())
print("\npIC50 range:", round(d["pIC50"].min(), 2), "to", round(d["pIC50"].max(), 2))

---
# Phase 2 - Molecular Curation

## 2.1 Standardise structures

Each SMILES string is parsed, desalted, reduced to its largest fragment,
sanitised, and rewritten in canonical form.

**Why.** The same compound may appear as a free base, a hydrochloride salt,
or written with different but equivalent SMILES. Without canonicalisation,
identical structures would be counted as distinct compounds.

In [ ]:
from rdkit import Chem
from rdkit.Chem import SaltRemover
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

remover = SaltRemover.SaltRemover()

def clean_smiles(smi):
    """Parse, desalt, keep largest fragment, sanitize, canonicalise."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = remover.StripMol(mol)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) > 1:
        mol = max(frags, key=lambda m: m.GetNumAtoms())
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol)

d["clean_smiles"] = d["canonical_smiles"].apply(clean_smiles)
print("Failed to parse:", d["clean_smiles"].isna().sum())
d = d.dropna(subset=["clean_smiles"])
print("Remaining rows:", len(d))

## 2.2 Aggregate replicate measurements

Compounds are frequently tested in several assays and publications. Records
are grouped by canonical structure and summarised by the **median** pIC50,
which is robust to outlying measurements.

The standard deviation across replicates is retained as a reliability
indicator: compounds where independent laboratories disagree by more than
one log unit are flagged.

In [ ]:
agg = (
    d.groupby("clean_smiles")
     .agg(
         chembl_id = ("molecule_chembl_id", "first"),
         pIC50     = ("pIC50", "median"),
         pIC50_std = ("pIC50", "std"),
         n_measure = ("pIC50", "size"),
     )
     .reset_index()
)

print("Unique compounds:", len(agg))
print("\nMeasurements per compound:")
print(agg["n_measure"].describe())

noisy = agg[(agg["n_measure"] > 1) & (agg["pIC50_std"] > 1)]
print("\nHigh-disagreement compounds (std > 1 log unit):", len(noisy))

## 2.3 Activity classification

| Class | pIC50 | IC50 equivalent |
|---|---|---|
| Active | >= 7 | <= 100 nM |
| Moderate | 5 - 7 | 100 nM - 10 uM |
| Inactive | < 5 | > 10 uM |

These thresholds follow standard medicinal-chemistry convention. Unlike a
single 10 uM cutoff - which would place almost every compound in one class -
they produce populated groups suitable for comparative statistics.

In [ ]:
def classify(p):
    if p >= 7:
        return "Active"
    elif p >= 5:
        return "Moderate"
    else:
        return "Inactive"

agg["Class"] = agg["pIC50"].apply(classify)
ORDER = ["Active", "Moderate", "Inactive"]

print(agg["Class"].value_counts().reindex(ORDER).to_string())
print("\nTotal compounds:", len(agg))

agg.to_csv("LSD1_clean.csv", index=False)
print("\nSaved: LSD1_clean.csv")

---
# Phase 3 - Physicochemical Descriptors

**Question:** are highly active LSD1 inhibitors chemically distinct from
weakly active ones in their bulk physicochemical properties?

## 3.1 Calculate descriptors

| Descriptor | Meaning |
|---|---|
| MW | Molecular weight |
| LogP | Lipophilicity (octanol/water partitioning) |
| TPSA | Topological polar surface area |
| HBD / HBA | Hydrogen bond donors / acceptors |
| RotB | Rotatable bonds (conformational flexibility) |
| Rings | Ring count |

In [ ]:
from rdkit.Chem import Descriptors, Crippen, Lipinski

df = pd.read_csv("LSD1_clean.csv")

def calc(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return pd.Series([np.nan] * 7)
    return pd.Series([
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Descriptors.TPSA(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
    ])

DESC = ["MW", "LogP", "TPSA", "HBD", "HBA", "RotB", "Rings"]
df[DESC] = df["clean_smiles"].apply(calc)
df = df.dropna(subset=DESC)

print("Compounds with descriptors:", len(df))
df.head()

## 3.2 Molecular size filtering

Initial inspection of this dataset revealed extreme outliers (MW up to
~3500 Da, LogP as low as -25). These correspond to **peptide-based
inhibitors** - unsurprising, since the natural substrate of LSD1 is the
histone H3 tail.

Peptides are legitimate LSD1 inhibitors but fall outside small-molecule
chemical space and are unsuitable for the docking protocol used in Phase 6.
They are therefore excluded, and the number removed is reported.

In [ ]:
print("BEFORE FILTERING")
print(df[["MW", "LogP"]].describe().round(2).to_string())

mask = df["MW"].between(150, 900) & df["LogP"].between(-5, 8)
removed = df[~mask]

print(f"\n\nRemoving {len(removed)} compounds ({100*len(removed)/len(df):.1f}%)")
if len(removed):
    print("\nProperties of removed compounds:")
    print(removed[["MW", "LogP"]].describe().round(2).to_string())
    print("\nBy class:")
    print(removed["Class"].value_counts().to_string())

df = df[mask].copy()
print(f"\n\nRETAINED: {len(df)} compounds")
print(df["Class"].value_counts().reindex(ORDER).to_string())

df.to_csv("LSD1_descriptors.csv", index=False)

## 3.3 Descriptor summary by activity class

Median values per class. This table is the numerical counterpart to the
plots that follow; dense scatter plots exaggerate apparent overlap, so the
medians are the record of reference.

In [ ]:
summary = df.groupby("Class")[DESC].median().round(2).reindex(ORDER)
print("MEDIAN DESCRIPTOR VALUES BY CLASS\n")
print(summary.to_string())

print("\n\nCOMPOUND COUNTS")
print(df["Class"].value_counts().reindex(ORDER).to_string())

## 3.4 Visualisation

Scatter plots relate pIC50 to molecular weight and lipophilicity; box plots
show the class distributions directly. Colour is supported by marker shape
so class identity is never carried by colour alone.

In [ ]:
import matplotlib.pyplot as plt

COLORS  = {"Active": "#2a78d6", "Moderate": "#eb6834", "Inactive": "#1baf7a"}
MARKERS = {"Active": "o",       "Moderate": "s",       "Inactive": "^"}
GRID, AXIS, INK, MUTED = "#e1e0d9", "#c3c2b7", "#0b0b0b", "#898781"

def style(ax):
    ax.set_facecolor("#fcfcfb")
    ax.grid(True, color=GRID, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), facecolor="#fcfcfb")
for ax, xvar, xlabel in zip(axes, ["MW", "LogP"],
                            ["Molecular Weight (Da)", "LogP (lipophilicity)"]):
    for cls in ORDER:
        sub = df[df["Class"] == cls]
        ax.scatter(sub[xvar], sub["pIC50"], s=11, alpha=0.40, linewidths=0,
                   c=COLORS[cls], marker=MARKERS[cls],
                   label=f"{cls} (n={len(sub)})", zorder=3)
    style(ax)
    ax.set_xlabel(xlabel, color=INK, fontsize=10)
    ax.set_ylabel("pIC50", color=INK, fontsize=10)
    ax.set_title(f"pIC50 vs {xvar}", color=INK, fontsize=12, loc="left")

leg = axes[0].legend(frameon=False, fontsize=9, markerscale=2.2, loc="lower right")
for h in leg.legend_handles:
    h.set_alpha(1.0)
plt.tight_layout()
plt.savefig("fig1_scatter.png", dpi=300, facecolor="#fcfcfb")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), facecolor="#fcfcfb")
for ax, var, label in zip(axes, ["MW", "LogP"], ["Molecular Weight (Da)", "LogP"]):
    data = [df.loc[df["Class"] == c, var] for c in ORDER]
    bp = ax.boxplot(data, patch_artist=True, notch=True, widths=0.55,
                    medianprops=dict(color=INK, linewidth=1.6),
                    flierprops=dict(marker=".", markersize=3,
                                    markerfacecolor=MUTED,
                                    markeredgecolor="none", alpha=0.4))
    for patch, cls in zip(bp["boxes"], ORDER):
        patch.set_facecolor(COLORS[cls])
        patch.set_alpha(0.65)
        patch.set_edgecolor(AXIS)
    ax.set_xticks(range(1, len(ORDER) + 1))
    ax.set_xticklabels(ORDER)
    style(ax)
    ax.set_ylabel(label, color=INK, fontsize=10)
    ax.set_title(f"{var} by activity class", color=INK, fontsize=12, loc="left")

plt.tight_layout()
plt.savefig("fig2_boxplots.png", dpi=300, facecolor="#fcfcfb")
plt.show()

## 3.5 Statistical testing with effect sizes

With large samples, trivially small differences become statistically
significant. A p-value indicates only that a difference exists, not that it
is chemically meaningful.

**Cliff's delta** is therefore reported alongside each p-value:

| abs(delta) | Interpretation |
|---|---|
| < 0.15 | negligible |
| 0.15 - 0.33 | small |
| 0.33 - 0.47 | medium |
| > 0.47 | large |

A descriptor is treated as meaningfully different only when it is both
statistically significant **and** has a non-negligible effect size.

In [ ]:
from scipy.stats import mannwhitneyu

act = df[df["Class"] == "Active"]
ina = df[df["Class"] == "Inactive"]
alpha = 0.05 / len(DESC)

def interpret(dlt):
    a = abs(dlt)
    if a < 0.15: return "negligible"
    if a < 0.33: return "small"
    if a < 0.47: return "medium"
    return "large"

rows = []
for v in DESC:
    u, p = mannwhitneyu(act[v], ina[v], alternative="two-sided")
    delta = (2 * u) / (len(act) * len(ina)) - 1
    rows.append({
        "Descriptor": v,
        "Act": round(act[v].median(), 2),
        "Inact": round(ina[v].median(), 2),
        "Diff": round(act[v].median() - ina[v].median(), 2),
        "p": f"{p:.1e}",
        "Cliff_d": round(delta, 3),
        "Effect": interpret(delta),
        "Meaningful": "YES" if (p < alpha and abs(delta) >= 0.15) else "no",
    })

res = pd.DataFrame(rows)
print(f"Active (n={len(act)}) vs Inactive (n={len(ina)})")
print(f"Bonferroni threshold: p < {alpha:.4f}\n")
print(res.to_string(index=False))
res.to_csv("table1_descriptor_stats.csv", index=False)

---
# Phase 4 - Fragment and Scaffold Analysis

Phase 3 asks whether bulk properties separate actives from inactives.
Phase 4 asks a sharper question: **which specific structural features are
associated with potency?**

## 4.1 Define functional group patterns

Functional groups are detected using SMARTS substructure patterns. The set
below covers common medicinal-chemistry fragments plus one of particular
relevance to this target.

**Cyclopropylamine** is included deliberately. Tranylcypromine-derived LSD1
inhibitors carry this group as a covalent warhead that forms an adduct with
the FAD cofactor. If the analysis is working, this fragment should emerge as
strongly associated with activity - a built-in positive control.

In [ ]:
FRAGMENTS = {
    "Cyclopropylamine": "[NX3][CX4H]1[CX4H2][CX4H2]1",   # TCP covalent warhead
    "Amide":            "[CX3](=[OX1])[NX3]",
    "Sulfonamide":      "[SX4](=[OX1])(=[OX1])[NX3]",
    "Urea":             "[NX3][CX3](=[OX1])[NX3]",
    "Ester":            "[CX3](=[OX1])[OX2H0][#6]",
    "CarboxylicAcid":   "[CX3](=[OX1])[OX2H1]",
    "Ketone":           "[#6][CX3](=[OX1])[#6]",
    "Ether":            "[OD2]([#6])[#6]",
    "Hydroxyl":         "[OX2H]",
    "PrimaryAmine":     "[NX3;H2][CX4]",
    "TertiaryAmine":    "[NX3]([#6])([#6])[#6]",
    "Nitrile":          "[NX1]#[CX2]",
    "Halogen":          "[F,Cl,Br,I]",
    "Benzene":          "c1ccccc1",
    "Pyridine":         "n1ccccc1",
    "Pyrimidine":       "c1cncnc1",
    "Imidazole":        "c1cnc[nH]1",
    "Triazole":         "c1nnc[nH]1",
    "Thiophene":        "c1ccsc1",
    "Furan":            "c1ccoc1",
    "Indole":           "c1ccc2[nH]ccc2c1",
    "Quinoline":        "c1ccc2ncccc2c1",
    "Piperidine":       "C1CCNCC1",
    "Piperazine":       "C1CNCCN1",
    "Pyrrolidine":      "C1CCNC1",
    "Morpholine":       "C1COCCN1",
}

patterns = {}
for name, sma in FRAGMENTS.items():
    p = Chem.MolFromSmarts(sma)
    if p is None:
        print("INVALID SMARTS:", name, sma)
    else:
        patterns[name] = p

print(f"{len(patterns)} valid SMARTS patterns compiled")

## 4.2 Build the fragment presence matrix

Each compound is tested for the presence of every fragment, producing a
boolean matrix of compounds x fragments.

In [ ]:
df = pd.read_csv("LSD1_descriptors.csv")
mols = [Chem.MolFromSmiles(s) for s in df["clean_smiles"]]
print("Molecules parsed:", sum(m is not None for m in mols), "/", len(mols))

for name, patt in patterns.items():
    df[f"frag_{name}"] = [
        bool(m.HasSubstructMatch(patt)) if m is not None else False
        for m in mols
    ]

frag_cols = [f"frag_{n}" for n in patterns]
print("Fragment columns added:", len(frag_cols))

## 4.3 Fragment frequency and enrichment

For each fragment, the percentage of compounds containing it is computed
within each activity class.

Two enrichment ratios are reported:

- **Active/Inactive** - values above 1 indicate the fragment is more common
  among potent compounds
- **Inactive/Active** - the inverse, as defined in the project workflow

Fragments present in fewer than 20 compounds are excluded; frequencies based
on very small counts are unstable.

In [ ]:
MIN_COUNT = 20
rows = []

for name in patterns:
    col = f"frag_{name}"
    total = int(df[col].sum())
    if total < MIN_COUNT:
        continue

    pct = {c: 100 * df.loc[df["Class"] == c, col].mean() for c in ORDER}
    a, i = pct["Active"], pct["Inactive"]

    rows.append({
        "Fragment": name,
        "N_total": total,
        "pct_Active": round(a, 1),
        "pct_Moderate": round(pct["Moderate"], 1),
        "pct_Inactive": round(i, 1),
        "Act/Inact": round(a / i, 2) if i > 0 else np.inf,
        "Inact/Act": round(i / a, 2) if a > 0 else np.inf,
    })

freq = pd.DataFrame(rows).sort_values("Act/Inact", ascending=False)
print("FRAGMENT FREQUENCY BY CLASS (%)\n")
print(freq.to_string(index=False))
freq.to_csv("table2_fragment_frequency.csv", index=False)

## 4.4 Activity-associated fragments

For each fragment, compounds are split into those containing it and those
without, and the two pIC50 distributions are compared.

$$\Delta pIC_{50} = \text{median}(\text{with fragment}) - \text{median}(\text{without fragment})$$

- Positive: the fragment is associated with increased potency
- Near zero: little apparent association
- Negative: the fragment is associated with reduced potency

Significance is assessed with the Mann-Whitney U test. Because many
fragments are tested simultaneously, p-values are corrected using the
**Benjamini-Hochberg** false discovery rate procedure, which is less
conservative than Bonferroni and more appropriate for exploratory screening.

Cliff's delta is again reported, since a significant p-value alone does not
establish that a difference matters.

In [ ]:
def benjamini_hochberg(pvals):
    """Return BH-adjusted p-values (same order as input)."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    # enforce monotonicity from the largest p downwards
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(ranked, 0, 1)
    return out

rows = []
for name in patterns:
    col = f"frag_{name}"
    with_f = df.loc[df[col], "pIC50"]
    without_f = df.loc[~df[col], "pIC50"]
    if len(with_f) < MIN_COUNT or len(without_f) < MIN_COUNT:
        continue

    u, p = mannwhitneyu(with_f, without_f, alternative="two-sided")
    delta = (2 * u) / (len(with_f) * len(without_f)) - 1

    rows.append({
        "Fragment": name,
        "N_with": len(with_f),
        "Median_with": round(with_f.median(), 2),
        "N_without": len(without_f),
        "Median_without": round(without_f.median(), 2),
        "dpIC50": round(with_f.median() - without_f.median(), 2),
        "p_raw": p,
        "Cliff_d": round(delta, 3),
        "Effect": interpret(delta),
    })

frag_stats = pd.DataFrame(rows)
frag_stats["p_adj"] = benjamini_hochberg(frag_stats["p_raw"])
frag_stats["Significant"] = np.where(
    (frag_stats["p_adj"] < 0.05) & (frag_stats["Cliff_d"].abs() >= 0.15),
    "YES", "no")

frag_stats = frag_stats.sort_values("dpIC50", ascending=False)
disp = frag_stats.copy()
disp["p_raw"] = disp["p_raw"].apply(lambda x: f"{x:.1e}")
disp["p_adj"] = disp["p_adj"].apply(lambda x: f"{x:.1e}")

print("ACTIVITY-ASSOCIATED FRAGMENTS (sorted by dpIC50)\n")
print(disp.to_string(index=False))
frag_stats.to_csv("table3_fragment_activity.csv", index=False)

print("\n\nFragments meeting both criteria (BH p<0.05 and |Cliff's d|>=0.15):")
print(frag_stats.loc[frag_stats["Significant"] == "YES", "Fragment"].tolist())

## 4.5 Distribution plots for the most informative fragments

Box plots of pIC50 with and without each of the six fragments showing the
largest absolute effect size.

In [ ]:
top = (frag_stats.reindex(frag_stats["Cliff_d"].abs()
                          .sort_values(ascending=False).index)
                 .head(6))

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5), facecolor="#fcfcfb")
for ax, (_, r) in zip(axes.ravel(), top.iterrows()):
    col = f"frag_{r['Fragment']}"
    data = [df.loc[~df[col], "pIC50"], df.loc[df[col], "pIC50"]]
    bp = ax.boxplot(data, patch_artist=True, notch=True, widths=0.55,
                    medianprops=dict(color=INK, linewidth=1.6),
                    flierprops=dict(marker=".", markersize=3,
                                    markerfacecolor=MUTED,
                                    markeredgecolor="none", alpha=0.35))
    for patch, c in zip(bp["boxes"], ["#1baf7a", "#2a78d6"]):
        patch.set_facecolor(c)
        patch.set_alpha(0.65)
        patch.set_edgecolor(AXIS)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"absent\n(n={r['N_without']})",
                        f"present\n(n={r['N_with']})"], fontsize=9)
    style(ax)
    ax.set_ylabel("pIC50", color=INK, fontsize=9)
    ax.set_title(f"{r['Fragment']}   dpIC50 = {r['dpIC50']:+.2f}",
                 color=INK, fontsize=11, loc="left")

plt.tight_layout()
plt.savefig("fig3_fragment_boxplots.png", dpi=300, facecolor="#fcfcfb")
plt.show()

## 4.6 Molecular context - scaffold analysis

A fragment does not act in isolation. For the fragment most strongly
associated with activity, the **Bemis-Murcko scaffolds** of the active
compounds containing it are extracted and the most frequent are depicted.

The Murcko scaffold is the ring system plus the linkers connecting rings,
with all side chains removed - the structural core of the molecule.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Draw

best = frag_stats.sort_values("Cliff_d", ascending=False).iloc[0]["Fragment"]
print("Fragment with the strongest positive association:", best)

sel = df[(df[f"frag_{best}"]) & (df["Class"] == "Active")].copy()
print("Active compounds containing it:", len(sel))

def murcko(smi):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smi)
    except Exception:
        return None

sel["scaffold"] = sel["clean_smiles"].apply(murcko)
sel = sel.dropna(subset=["scaffold"])
sel = sel[sel["scaffold"] != ""]

counts = sel["scaffold"].value_counts().head(8)
print(f"\nDistinct scaffolds: {sel['scaffold'].nunique()}")
print("\nTop scaffolds by frequency:")
for s, c in counts.items():
    med = sel.loc[sel["scaffold"] == s, "pIC50"].median()
    print(f"  n={c:<4} median pIC50={med:.2f}   {s}")

scaf_mols, legends = [], []
for s, c in counts.items():
    m = Chem.MolFromSmiles(s)
    if m is not None:
        med = sel.loc[sel["scaffold"] == s, "pIC50"].median()
        scaf_mols.append(m)
        legends.append(f"n={c}, median pIC50={med:.2f}")

img = Draw.MolsToGridImage(scaf_mols, legends=legends, molsPerRow=4,
                           subImgSize=(300, 260), returnPNG=False)
img.save("fig4_scaffolds.png")
img

## 4.7 Save the annotated dataset

The fragment matrix is written out for use in Phase 5.

In [ ]:
df.to_csv("LSD1_fragments.csv", index=False)
print("Saved LSD1_fragments.csv  ", df.shape)

print("\nFiles produced:")
for f in sorted(os.listdir(".")):
    if f.endswith((".csv", ".png")):
        print("  ", f)

---
## Interpretation checklist

Before moving to Phase 5, be able to answer:

1. Which fragments are **positively** associated with potency, and does
   cyclopropylamine appear among them as expected?
2. Do any fragments show a large p-value effect but a **negligible** Cliff's
   delta? Those must not be reported as meaningful.
3. Are the active compounds dominated by **one scaffold** or spread across
   several? This determines whether the similarity search in Phase 5 should
   expect a single chemotype or multiple.

---

**Next: Phase 5** - Morgan fingerprint similarity analysis, Enamine
diversity-set screening, and the approved-drug repurposing screen.

---
# Phase 5 - Molecular Similarity and Library Screening

## 5.1 Morgan fingerprints

Each molecule is encoded as a 2048-bit Morgan (ECFP4) fingerprint -
a binary vector recording which circular substructures are present.

Similarity between two molecules is the **Tanimoto coefficient**: the
number of bits set in both, divided by the number set in either. It runs
from 0 (nothing in common) to 1 (identical fingerprint).

In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

ORDER = ["Active", "Moderate", "Inactive"]
df = pd.read_csv("LSD1_fragments.csv")
print("Loaded:", df.shape)
print(df["Class"].value_counts().reindex(ORDER).to_string())

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def fp_of(smi):
    m = Chem.MolFromSmiles(smi)
    return gen.GetFingerprint(m) if m is not None else None

df["fp"] = df["clean_smiles"].apply(fp_of)
df = df[df["fp"].notna()].reset_index(drop=True)
print("\nFingerprints generated:", len(df))

## 5.2 Are the active compounds structurally similar to each other?

This determines how Phase 5 should proceed. Two possibilities:

- **High mutual similarity** - the actives form one chemical series, and a
  similarity search will find close analogues of a single chemotype.
- **Low mutual similarity** - the actives span several distinct scaffolds,
  and any single query molecule will only retrieve part of the space.

Two distributions are compared: similarity among actives, and similarity
between actives and inactives. If the first is not clearly higher than the
second, structure alone does not separate potency.

In [ ]:
act_fps = list(df.loc[df["Class"] == "Active", "fp"])
ina_fps = list(df.loc[df["Class"] == "Inactive", "fp"])
print(f"Actives: {len(act_fps)}   Inactives: {len(ina_fps)}")

# Active vs Active (upper triangle only)
aa = []
for i in range(1, len(act_fps)):
    aa.extend(DataStructs.BulkTanimotoSimilarity(act_fps[i], act_fps[:i]))
aa = np.array(aa)

# Active vs Inactive (all cross pairs)
ai = []
for f in act_fps:
    ai.extend(DataStructs.BulkTanimotoSimilarity(f, ina_fps))
ai = np.array(ai)

print(f"\nActive-Active pairs   : {len(aa):,}   mean {aa.mean():.3f}   median {np.median(aa):.3f}")
print(f"Active-Inactive pairs : {len(ai):,}   mean {ai.mean():.3f}   median {np.median(ai):.3f}")

for t in (0.5, 0.7, 0.85, 0.9):
    print(f"  pairs >= {t}:  Act-Act {100*(aa>=t).mean():6.2f}%   "
          f"Act-Inact {100*(ai>=t).mean():6.2f}%")

In [ ]:
import matplotlib.pyplot as plt

GRID, AXIS, INK, MUTED = "#e1e0d9", "#c3c2b7", "#0b0b0b", "#898781"

fig, ax = plt.subplots(figsize=(8.5, 4.8), facecolor="#fcfcfb")
bins = np.linspace(0, 1, 60)
ax.hist(aa, bins=bins, density=True, alpha=0.60, color="#2a78d6",
        label=f"Active-Active (n={len(aa):,})")
ax.hist(ai, bins=bins, density=True, alpha=0.60, color="#1baf7a",
        label=f"Active-Inactive (n={len(ai):,})")
ax.set_facecolor("#fcfcfb")
ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)
ax.set_xlabel("Tanimoto similarity", color=INK)
ax.set_ylabel("Density", color=INK)
ax.set_title("Pairwise structural similarity", color=INK, fontsize=12, loc="left")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig("fig5_similarity_distributions.png", dpi=300, facecolor="#fcfcfb")
plt.show()

## 5.3 Scaffold diversity among the actives

The number of distinct Bemis-Murcko scaffolds, and how much of the active
set the most common scaffolds account for. A small number of scaffolds
covering most actives indicates a few dominant chemical series.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold

def murcko(smi):
    try:
        s = MurckoScaffold.MurckoScaffoldSmiles(smi)
        return s if s else None
    except Exception:
        return None

act = df[df["Class"] == "Active"].copy()
act["scaffold"] = act["clean_smiles"].apply(murcko)
act = act.dropna(subset=["scaffold"])

vc = act["scaffold"].value_counts()
print(f"Active compounds      : {len(act)}")
print(f"Distinct scaffolds    : {len(vc)}")
print(f"Singleton scaffolds   : {(vc == 1).sum()}  ({100*(vc==1).sum()/len(vc):.0f}% of scaffolds)")
print(f"\nCoverage by most common scaffolds:")
cum = vc.cumsum() / len(act) * 100
for k in (1, 5, 10, 20, 50):
    if k <= len(vc):
        print(f"  top {k:>3} scaffolds cover {cum.iloc[k-1]:5.1f}% of actives")

## 5.4 Download the screening library (ZINC)

Enamine's Discovery Diversity Set requires registered access. An equivalent
set of commercially available compounds is obtained from **ZINC**, which
aggregates supplier catalogues - including Enamine's.

Tranches spanning drug-like molecular weight and logP ranges are downloaded
and pooled. A diversity set would be a poor choice here: it is constructed
to minimise near-neighbours, and would return almost nothing at the
similarity thresholds used below.

In [ ]:
import requests, io

# files.docking.org returns 403 to the default python-requests User-Agent.
HDRS = {"User-Agent": "Mozilla/5.0"}

TRANCHES = ["CBAA", "DCAA", "DCAB", "EBAB", "ECAB",
            "FCAB", "FDAB", "GCAB", "GDAB", "HDAB"]

LIB = "screening_library.csv"

if not os.path.exists(LIB):
    frames = []
    for t in TRANCHES:
        url = f"https://files.docking.org/2D/{t[:2]}/{t}.smi"
        try:
            r = requests.get(url, timeout=180, headers=HDRS)
            if r.status_code != 200:
                print(f"  {t}: HTTP {r.status_code}")
                continue
            if r.text.lstrip().startswith("<"):
                print(f"  {t}: received HTML, not SMILES")
                continue
            part = pd.read_csv(io.StringIO(r.text), sep=r"\s+")
            frames.append(part)
            print(f"  {t}: {len(part):,} compounds")
        except Exception as e:
            print(f"  {t}: {type(e).__name__} - {e}")

    if not frames:
        raise RuntimeError(
            "No tranches downloaded - see the HTTP codes above.\n"
            "  403  the User-Agent header was rejected\n"
            "  404  that tranche code no longer exists\n"
            "  timeout/connection  files.docking.org unreachable from here\n\n"
            "Fallbacks: download the .smi files in a browser and upload them "
            "to this folder, or skip the ZINC screen and use the drug "
            "repurposing library alone."
        )

    lib = pd.concat(frames, ignore_index=True).drop_duplicates("smiles")
    lib.to_csv(LIB, index=False)
else:
    lib = pd.read_csv(LIB)

print(f"\nLibrary compounds: {len(lib):,}")
lib.head()

## 5.5 Download the drug repurposing library

The **Broad Drug Repurposing Hub** catalogues approved and clinical-stage
drugs with structures and mechanism-of-action annotations. Screening it
alongside the ZINC library allows a direct comparison: do novel compounds
or existing drugs rank better against LSD1?

In [ ]:
BASE = "https://s3.amazonaws.com/data.clue.io/repurposing/downloads"

def load_broad(fname):
    txt = requests.get(f"{BASE}/{fname}", timeout=180,
                       headers={"User-Agent": "Mozilla/5.0"}).text
    lines = [l for l in txt.splitlines() if not l.startswith("!")]
    return pd.read_csv(io.StringIO("\n".join(lines)), sep="\t")

samples = load_broad("repurposing_samples_20200324.txt")
drugs   = load_broad("repurposing_drugs_20200324.txt")

print("samples:", samples.shape, "| columns:", list(samples.columns)[:8])
print("drugs  :", drugs.shape,   "| columns:", list(drugs.columns)[:8])

dl = (samples[["pert_iname", "smiles"]]
      .dropna()
      .drop_duplicates("pert_iname")
      .reset_index(drop=True))

ann = drugs[["pert_iname", "clinical_phase", "moa", "target"]].drop_duplicates("pert_iname")
dl = dl.merge(ann, on="pert_iname", how="left")

print(f"\nUnique drugs with SMILES: {len(dl):,}")
print(dl["clinical_phase"].value_counts().head(8).to_string())
dl.to_csv("drug_library.csv", index=False)
dl.head()

## 5.6 Similarity screening with a threshold sweep

Every library compound is compared against every active compound; its score
is the **maximum** Tanimoto similarity to any active.

The project brief specifies a 0.90 cutoff. Rather than reporting a single
number, the hit count is tabulated across a range of thresholds. This
characterises the library properly and identifies the highest cutoff that
still yields a workable number of compounds.

In [ ]:
def screen(smiles_list, query_fps, label, chunk=5000):
    """Max Tanimoto similarity of each library compound to any query."""
    best = np.zeros(len(smiles_list), dtype=float)
    valid = np.zeros(len(smiles_list), dtype=bool)
    for start in range(0, len(smiles_list), chunk):
        block = smiles_list[start:start + chunk]
        for j, smi in enumerate(block):
            m = Chem.MolFromSmiles(smi)
            if m is None:
                continue
            f = gen.GetFingerprint(m)
            best[start + j] = max(DataStructs.BulkTanimotoSimilarity(f, query_fps))
            valid[start + j] = True
        print(f"  {label}: {min(start+chunk, len(smiles_list)):,}/{len(smiles_list):,}")
    return best, valid

zinc_sim, zinc_ok = screen(lib["smiles"].tolist(), act_fps, "ZINC")
lib["max_sim"] = zinc_sim
lib = lib[zinc_ok].copy()

drug_sim, drug_ok = screen(dl["smiles"].tolist(), act_fps, "drugs")
dl["max_sim"] = drug_sim
dl = dl[drug_ok].copy()

In [ ]:
print("THRESHOLD SWEEP\n")
print(f"{'cutoff':>8}  {'ZINC hits':>12}  {'Drug hits':>12}")
print("-" * 36)
for t in (0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50):
    print(f"{t:>8.2f}  {int((lib['max_sim']>=t).sum()):>12,}  "
          f"{int((dl['max_sim']>=t).sum()):>12,}")

print(f"\nZINC  max similarity observed: {lib['max_sim'].max():.3f}")
print(f"Drugs max similarity observed: {dl['max_sim'].max():.3f}")

print("\n\nTOP 15 DRUG HITS")
cols = ["pert_iname", "max_sim", "clinical_phase", "moa"]
print(dl.nlargest(15, "max_sim")[cols].to_string(index=False))

## 5.7 Select the working cutoff and cluster the hits

A catalogue library contains families of close analogues, so a raw hit list
is highly redundant. Hits are clustered with the **Butina** algorithm and one
representative - the most similar to an active - is taken from each cluster.

This yields a structurally diverse shortlist for docking rather than
many variations on one molecule.

In [ ]:
from rdkit.ML.Cluster import Butina

# Pick the highest cutoff that still returns a usable number of compounds
CUTOFF = None
for t in (0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60):
    if (lib["max_sim"] >= t).sum() >= 50:
        CUTOFF = t
        break
if CUTOFF is None:
    CUTOFF = 0.60
print("Working similarity cutoff:", CUTOFF)

hits = lib[lib["max_sim"] >= CUTOFF].copy().reset_index(drop=True)
print("Raw hits:", len(hits))

hit_fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in hits["smiles"]]

dists = []
for i in range(1, len(hit_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(hit_fps[i], hit_fps[:i])
    dists.extend(1.0 - np.array(sims))

clusters = Butina.ClusterData(dists, len(hit_fps), 0.35, isDistData=True)
print("Clusters:", len(clusters))

reps = []
for cl in clusters:
    sub = hits.iloc[list(cl)]
    reps.append(sub["max_sim"].idxmax())
zinc_hits = hits.loc[reps].sort_values("max_sim", ascending=False).reset_index(drop=True)

print("Cluster representatives:", len(zinc_hits))
zinc_hits.to_csv("hits_zinc.csv", index=False)
zinc_hits.head(10)

In [ ]:
DRUG_CUTOFF = 0.40
drug_hits = (dl[dl["max_sim"] >= DRUG_CUTOFF]
             .sort_values("max_sim", ascending=False)
             .head(100)
             .reset_index(drop=True))

print(f"Drug hits at >= {DRUG_CUTOFF}: {len(drug_hits)}")
drug_hits.to_csv("hits_drugs.csv", index=False)
print(drug_hits[["pert_iname", "max_sim", "clinical_phase", "moa"]].head(20).to_string(index=False))

## 5.8 Assemble the docking set

Four groups are docked together:

| Group | Purpose |
|---|---|
| **Known actives** | Should rank highly if docking works |
| **Known inactives** | Should rank poorly - the negative control |
| **ZINC hits** | Novel candidate compounds |
| **Drug hits** | Repurposing candidates |

Including known actives and inactives is essential: without them the
docking scores cannot be validated, and the enrichment calculation in
Phase 6 would be impossible.

In [ ]:
sel_act = (df[df["Class"] == "Active"]
           .nlargest(N_ACT, "pIC50")[["clean_smiles", "pIC50"]]
           .assign(group="active"))
sel_ina = (df[df["Class"] == "Inactive"]
           .sample(min(N_INA, (df["Class"] == "Inactive").sum()), random_state=42)
           [["clean_smiles", "pIC50"]]
           .assign(group="inactive"))

sel_zinc = (zinc_hits.head(N_ZINC)
            .rename(columns={"smiles": "clean_smiles"})[["clean_smiles"]]
            .assign(pIC50=np.nan, group="zinc_hit"))
sel_drug = (drug_hits.head(N_DRUG)
            .rename(columns={"smiles": "clean_smiles"})[["clean_smiles"]]
            .assign(pIC50=np.nan, group="drug_hit"))

dock_set = pd.concat([sel_act, sel_ina, sel_zinc, sel_drug], ignore_index=True)
dock_set = dock_set.drop_duplicates("clean_smiles").reset_index(drop=True)
dock_set["lig_id"] = [f"L{i:04d}" for i in range(len(dock_set))]

print(dock_set["group"].value_counts().to_string())
print("\nTotal ligands to dock:", len(dock_set))
dock_set.to_csv("docking_set.csv", index=False)

---
# Phase 6 - Molecular Docking

## 6.1 Install docking software

- **AutoDock Vina** - the docking engine
- **Meeko** - ligand preparation (torsions, atom typing, PDBQT output)
- **PDBFixer / OpenMM** - repair of missing protein atoms
- **Open Babel** - receptor PDBQT conversion
- **PLIP** - protein-ligand interaction profiling
- **py3Dmol** - in-notebook structure visualisation

Installation takes several minutes.

In [ ]:
# The `vina` PyPI package compiles from source and needs Boost + SWIG, which
# fails on Colab. The official precompiled Linux binary is used instead -
# identical program, no build step.

# meeko 0.8.0 ships with no declared dependencies, so gemmi is
# installed explicitly or import meeko fails.
!pip install -q meeko gemmi pdbfixer openmm plip py3Dmol biopython
!apt-get -qq install -y openbabel > /dev/null 2>&1

import os, stat, subprocess, requests

VINA = "/content/vina"
VINA_URL = ("https://github.com/ccsb-scripps/AutoDock-Vina/releases/"
            "download/v1.2.7/vina_1.2.7_linux_x86_64")

if not os.path.exists(VINA):
    r = requests.get(VINA_URL, timeout=600, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    with open(VINA, "wb") as fh:
        fh.write(r.content)
    os.chmod(VINA, os.stat(VINA).st_mode | stat.S_IEXEC)
    print(f"downloaded vina ({len(r.content):,} bytes)")

print(subprocess.run([VINA, "--version"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["obabel", "-V"], capture_output=True, text=True).stdout.strip())

import meeko, plip, py3Dmol
print("meeko, plip, py3Dmol imported OK")

## 6.2 Download and inspect the structure

**6TUY** - human LSD1/CoREST with FAD and the reversible quinazoline
inhibitor MC4106 (ligand code **NY8**), 2.60 A.

The contents are listed before any processing, so that what is kept and what
is discarded is explicit.

In [ ]:
PDB_ID   = "6TUY"
LIG_CODE = "NY8"

if not os.path.exists(f"{PDB_ID}.pdb"):
    r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb",
                     timeout=180, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    open(f"{PDB_ID}.pdb", "w").write(r.text)

lines = open(f"{PDB_ID}.pdb").read().splitlines()
print([l for l in lines if l.startswith("TITLE")][0])

from collections import Counter
het = Counter(l[17:20].strip() for l in lines if l.startswith("HETATM"))
chains = Counter(l[21] for l in lines if l.startswith("ATOM"))

print("\nProtein chains (atom counts):", dict(chains))
print("\nHeterogens present:")
for k, v in het.most_common():
    tag = ""
    if k == "FAD":      tag = "  <-- COFACTOR, MUST KEEP"
    elif k == LIG_CODE: tag = "  <-- crystal ligand (reference pose)"
    elif k == "HOH":    tag = "  (water, removed)"
    print(f"   {k:<5} {v:>6} atoms{tag}")

## 6.3 Prepare the receptor - keeping FAD

**This is the step where LSD1 docking usually goes wrong.**

LSD1 is a flavin-dependent enzyme and FAD sits at the base of the inhibitor
binding site. Standard cleanup routines classify FAD as a heterogen and
delete it along with waters and buffer components. If that happens the
pocket becomes an open cavity and every docking score is meaningless -
with no error raised.

The protein and FAD are therefore handled separately: PDBFixer repairs the
protein only, and FAD is reattached afterwards. The final file is checked
for FAD before docking proceeds.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# 1. split protein / FAD / crystal ligand
prot_lines = [l for l in lines if l.startswith(("ATOM", "TER"))]
fad_lines  = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == "FAD"]
lig_lines  = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == LIG_CODE]

print(f"protein atoms {len(prot_lines)} | FAD atoms {len(fad_lines)} | ligand atoms {len(lig_lines)}")
assert fad_lines, "FAD not found - stop and check the structure"
assert lig_lines, "crystal ligand not found"

open("protein_only.pdb", "w").write("\n".join(prot_lines) + "\nEND\n")
open("fad.pdb", "w").write("\n".join(fad_lines) + "\nEND\n")
open("crystal_ligand.pdb", "w").write("\n".join(lig_lines) + "\nEND\n")

# 2. repair the protein
fixer = PDBFixer(filename="protein_only.pdb")
fixer.findMissingResidues()
fixer.findMissingAtoms()
print("\nmissing residues:", len(fixer.missingResidues),
      "| missing atoms:", len(fixer.missingAtoms))
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)
PDBFile.writeFile(fixer.topology, fixer.positions,
                  open("protein_fixed.pdb", "w"), keepIds=True)

# 3. reattach FAD
fixed = [l for l in open("protein_fixed.pdb").read().splitlines()
         if l.startswith(("ATOM", "TER"))]
open("receptor.pdb", "w").write("\n".join(fixed + fad_lines) + "\nEND\n")

check = open("receptor.pdb").read()
print("\nFAD present in receptor.pdb:", "FAD" in check)
assert "FAD" in check, "FAD was lost during preparation"

In [ ]:
# Convert to PDBQT (rigid receptor)
!obabel receptor.pdb -O receptor.pdbqt -xr -p 7.4 2> /dev/null
n = sum(1 for l in open("receptor.pdbqt") if l.startswith(("ATOM", "HETATM")))
print("receptor.pdbqt atoms:", n)
print("FAD retained in PDBQT:", "FAD" in open("receptor.pdbqt").read())

## 6.4 Define the docking box

The search box is centred on the centroid of the crystal ligand, which marks
the experimentally observed binding site. A 22 A cube comfortably encloses
the pocket without inflating the search space.

In [ ]:
coords = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                   for l in lig_lines])
center = coords.mean(axis=0)
extent = coords.max(axis=0) - coords.min(axis=0)

BOX = [22.0, 22.0, 22.0]
CENTER = [round(float(c), 3) for c in center]

print("crystal ligand centroid :", CENTER)
print("crystal ligand extent   :", [round(float(e), 1) for e in extent])
print("docking box size        :", BOX)

### Visual verification of the prepared receptor and box

Before committing to a long docking run, confirm three things by eye:

1. **FAD is present** (yellow sticks) and sits at the base of the pocket
2. **The crystal ligand** (magenta) lies directly against FAD
3. **The docking box** (blue) encloses the ligand with room to spare

If the magenta ligand and the yellow cofactor are far apart, or the box
does not contain the ligand, the setup is wrong - stop and diagnose it
here rather than after two hours of docking.

In [ ]:
import py3Dmol

v3 = py3Dmol.view(width=900, height=620)

# receptor cartoon
v3.addModel(open("receptor.pdb").read(), "pdb")
v3.setStyle({"cartoon": {"color": "spectrum"}})

# FAD cofactor
v3.addStyle({"resn": "FAD"},
            {"stick": {"colorscheme": "yellowCarbon", "radius": 0.18}})

# crystal ligand (reference pose)
v3.addModel(open("crystal_ligand.pdb").read(), "pdb")
v3.setStyle({"model": -1},
            {"stick": {"colorscheme": "magentaCarbon", "radius": 0.22}})

# docking search box
v3.addBox({
    "center": {"x": CENTER[0], "y": CENTER[1], "z": CENTER[2]},
    "dimensions": {"w": BOX[0], "h": BOX[1], "d": BOX[2]},
    "color": "blue", "opacity": 0.18,
})

v3.zoomTo({"model": -1})
print("yellow = FAD   magenta = crystal ligand   blue = docking box")
print("box centre:", CENTER, " size:", BOX)
v3.show()

## 6.5 Prepare ligands

Meeko converts each SMILES into a 3D PDBQT file with rotatable bonds
assigned. Molecules that fail embedding are reported and skipped.

In [ ]:
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy
from collections import Counter

os.makedirs("ligands", exist_ok=True)
dock_set = pd.read_csv("docking_set.csv")

def make_pdbqt(smi, path):
    """SMILES -> 3D PDBQT. Returns (ok, reason)."""
    try:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return False, "unparseable SMILES"

        # Library compounds are often salts or mixtures; Meeko needs one fragment.
        frags = Chem.GetMolFrags(m, asMols=True, sanitizeFrags=True)
        if len(frags) > 1:
            m = max(frags, key=lambda x: x.GetNumHeavyAtoms())

        m = Chem.AddHs(m)
        if AllChem.EmbedMolecule(m, randomSeed=0xC0FFEE) != 0:
            return False, "3D embedding failed"
        try:
            AllChem.MMFFOptimizeMolecule(m, maxIters=400)
        except Exception:
            pass

        setups = MoleculePreparation().prepare(m)
        if not setups:
            return False, "meeko returned nothing"
        txt, ok, err = PDBQTWriterLegacy.write_string(setups[0])
        if not ok:
            return False, f"pdbqt write failed: {err}"
        with open(path, "w") as fh:
            fh.write(txt)
        return True, "ok"
    except Exception as e:
        return False, type(e).__name__

ok_rows, reasons = [], Counter()
for _, r in dock_set.iterrows():
    p = f"ligands/{r['lig_id']}.pdbqt"
    if os.path.exists(p):
        ok_rows.append(r); reasons["already prepared"] += 1; continue
    ok, why = make_pdbqt(r["clean_smiles"], p)
    if ok:
        ok_rows.append(r); reasons["ok"] += 1
    else:
        reasons[why] += 1

prepared = pd.DataFrame(ok_rows).reset_index(drop=True)
print(f"prepared {len(prepared)} of {len(dock_set)}\n")
print("outcomes:")
for k, v in reasons.most_common():
    print(f"   {k:<30} {v}")
print("\nby group:")
print(prepared["group"].value_counts().to_string())
prepared.to_csv("docking_set_prepared.csv", index=False)

## 6.6 Protocol validation by redocking

Before docking unknown compounds, the protocol is tested on a molecule whose
answer is already known: the crystal ligand is extracted, stripped of its
coordinates, and docked back into the receptor.

The predicted pose is compared with the crystallographic pose by RMSD.

| RMSD | Interpretation |
|---|---|
| < 2.0 A | Protocol reproduces the experimental pose - acceptable |
| 2.0 - 3.0 A | Marginal; interpret results cautiously |
| > 3.0 A | Protocol is not reliable for this site |

**Any docking study without this check is uninterpretable.** It is the only
direct evidence that the setup - box, receptor preparation, cofactor
placement - is correct.

In [ ]:
import subprocess

def run_vina(ligand, out, exhaustiveness=8, n_poses=9, seed=42):
    """Run one Vina docking job. Returns the CompletedProcess."""
    cmd = [VINA,
           "--receptor", "receptor.pdbqt",
           "--ligand", ligand,
           "--out", out,
           "--center_x", f"{CENTER[0]}",
           "--center_y", f"{CENTER[1]}",
           "--center_z", f"{CENTER[2]}",
           "--size_x", f"{BOX[0]}",
           "--size_y", f"{BOX[1]}",
           "--size_z", f"{BOX[2]}",
           "--exhaustiveness", str(exhaustiveness),
           "--num_modes", str(n_poses),
           "--seed", str(seed)]
    return subprocess.run(cmd, capture_output=True, text=True)


def best_score(path):
    """Best (first) affinity from a Vina output PDBQT, in kcal/mol."""
    try:
        for line in open(path):
            if line.startswith("REMARK VINA RESULT:"):
                return float(line.split()[3])
    except FileNotFoundError:
        pass
    return None


# The crystal ligand's bond orders are not stored in the PDB file, so the
# authoritative SMILES is fetched from the RCSB chemical component dictionary.
cc = requests.get(f"https://data.rcsb.org/rest/v1/core/chemcomp/{LIG_CODE}",
                  timeout=120, headers={"User-Agent": "Mozilla/5.0"}).json()
LIG_SMILES = cc["rcsb_chem_comp_descriptor"]["SMILES"]
print(f"{LIG_CODE}: {cc['chem_comp']['name'][:70]}")
print("SMILES:", LIG_SMILES)

redock_ok, why = make_pdbqt(LIG_SMILES, "ligands/REDOCK.pdbqt")
print("redock ligand prepared:", redock_ok, "-", why)

if redock_ok:
    proc = run_vina("ligands/REDOCK.pdbqt", "redock_out.pdbqt",
                    exhaustiveness=16)
    if proc.returncode != 0:
        print("VINA FAILED (exit", proc.returncode, ")")
        print(proc.stdout[-1500:])
        print(proc.stderr[-1500:])
    else:
        print("\nredock score:", best_score("redock_out.pdbqt"), "kcal/mol")

In [ ]:
# RMSD between the redocked pose and the crystallographic pose
!obabel redock_out.pdbqt -O redock_pose.pdb -f 1 -l 1 2> /dev/null

pose = Chem.MolFromPDBFile("redock_pose.pdb", sanitize=False, removeHs=True)
xtal = Chem.MolFromPDBFile("crystal_ligand.pdb", sanitize=False, removeHs=True)

if pose and xtal and pose.GetNumAtoms() == xtal.GetNumAtoms():
    a = pose.GetConformer().GetPositions()
    b = xtal.GetConformer().GetPositions()
    rmsd = float(np.sqrt(((a - b) ** 2).sum(axis=1).mean()))
    print(f"Redocking RMSD: {rmsd:.2f} A")
    print("VERDICT:", "PASS" if rmsd < 2.0 else
                      "MARGINAL" if rmsd < 3.0 else "FAIL")
else:
    print("Atom counts differ - compare poses visually in 6.9 instead.")
    print("pose:", pose.GetNumAtoms() if pose else None,
          " xtal:", xtal.GetNumAtoms() if xtal else None)

## 6.7 Dock the full set

Each ligand is docked using the affinity maps already computed. Results are
appended to disk after every molecule, so an interrupted session can be
resumed without losing completed work.

Expect roughly 10-20 seconds per ligand. Several hundred ligands therefore
takes one to two hours - leave the cell running.

In [ ]:
import time, csv

RESULTS = "docking_results.csv"
done = set()
if os.path.exists(RESULTS):
    done = set(pd.read_csv(RESULTS)["lig_id"])
    print("resuming - already docked:", len(done))

prepared = pd.read_csv("docking_set_prepared.csv")
todo = prepared[~prepared["lig_id"].isin(done)]
print("to dock:", len(todo))

write_header = not os.path.exists(RESULTS)
t0 = time.time()
fails = 0

with open(RESULTS, "a", newline="") as fh:
    w = csv.writer(fh)
    if write_header:
        w.writerow(["lig_id", "group", "pIC50", "smiles", "score"])

    for i, (_, r) in enumerate(todo.iterrows(), 1):
        lid = r["lig_id"]
        out = f"ligands/{lid}_out.pdbqt"
        score = None
        try:
            proc = run_vina(f"ligands/{lid}.pdbqt", out,
                            exhaustiveness=EXHAUSTIVENESS, n_poses=5)
            if proc.returncode == 0:
                score = best_score(out)
            else:
                fails += 1
                if fails <= 3:
                    print(f"  {lid} failed:", proc.stderr.strip()[-200:])
        except Exception as e:
            fails += 1
            print("  ", lid, type(e).__name__)

        w.writerow([lid, r["group"], r["pIC50"], r["clean_smiles"],
                    score if score is not None else float("nan")])
        fh.flush()

        if i % 20 == 0:
            el = time.time() - t0
            print(f"  {i}/{len(todo)}   {el/i:.1f}s/ligand   "
                  f"ETA {(len(todo)-i)*el/i/60:.0f} min")

print(f"\ndocking complete - {fails} failures")

## 6.8 Enrichment analysis

The central question: **does docking rank known actives above known
inactives?**

If it does, the scores carry real signal and the novel hits are worth
taking seriously. If it does not, the docking protocol has no discriminative
power for this target and the ranking of new compounds means nothing.

Reported honestly either way - a negative result that is correctly
diagnosed is a legitimate finding.

In [ ]:
res = pd.read_csv("docking_results.csv").dropna(subset=["score"])
res = res.sort_values("score")          # more negative = better
res["rank"] = range(1, len(res) + 1)

print("Docked:", len(res))
print("\nScore by group (kcal/mol):")
print(res.groupby("group")["score"].describe()[["count","mean","50%","min"]].round(2).to_string())

n_act = (res["group"] == "active").sum()
n_ina = (res["group"] == "inactive").sum()

print("\n\nENRICHMENT")
print(f"{'top %':>7} {'n':>5} {'actives':>9} {'inactives':>11} {'EF':>7}")
print("-" * 44)
for pct in (1, 5, 10, 20):
    k = max(1, int(len(res) * pct / 100))
    top = res.head(k)
    a = (top["group"] == "active").sum()
    i = (top["group"] == "inactive").sum()
    expected = n_act * k / len(res)
    ef = a / expected if expected > 0 else np.nan
    print(f"{pct:>6}% {k:>5} {a:>9} {i:>11} {ef:>7.2f}")

k5 = max(1, int(len(res) * 0.05))
pct_in_top5 = 100 * (res.head(k5)["group"] == "active").sum() / n_act
print(f"\n{pct_in_top5:.1f}% of known actives fall in the top 5% of the ranking")

from scipy.stats import mannwhitneyu
sa = res.loc[res["group"] == "active", "score"]
si = res.loc[res["group"] == "inactive", "score"]
u, p = mannwhitneyu(sa, si, alternative="less")
print(f"Actives score better than inactives: p = {p:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8), facecolor="#fcfcfb")
GROUPS = ["active", "inactive", "zinc_hit", "drug_hit"]
COLS   = ["#2a78d6", "#1baf7a", "#eb6834", "#eda100"]
data = [res.loc[res["group"] == g, "score"] for g in GROUPS]
data = [d for d in data if len(d)]
labs = [g for g, d in zip(GROUPS, [res.loc[res["group"]==g,"score"] for g in GROUPS]) if len(d)]

bp = ax.boxplot(data, patch_artist=True, notch=True, widths=0.55,
                medianprops=dict(color=INK, linewidth=1.6))
for patch, c in zip(bp["boxes"], COLS):
    patch.set_facecolor(c); patch.set_alpha(0.65); patch.set_edgecolor(AXIS)
ax.set_xticks(range(1, len(labs) + 1)); ax.set_xticklabels(labs)
ax.set_facecolor("#fcfcfb"); ax.grid(True, color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)
ax.set_ylabel("Vina score (kcal/mol)", color=INK)
ax.set_title("Docking scores by compound group", color=INK, fontsize=12, loc="left")
plt.tight_layout()
plt.savefig("fig6_docking_scores.png", dpi=300, facecolor="#fcfcfb")
plt.show()

print("\nTOP 20 OVERALL")
print(res.head(20)[["rank","lig_id","group","score","pIC50"]].to_string(index=False))

## 6.9 Interaction analysis of the top compounds (PLIP)

For the five best-scoring compounds, the docked pose is combined with the
receptor and analysed with **PLIP**, which detects hydrogen bonds,
hydrophobic contacts, pi-stacking, salt bridges and halogen bonds.

Interactions involving **FAD** are of particular interest: contact with the
cofactor indicates binding in the catalytic site rather than at a surface
pocket.

In [ ]:
import subprocess
from plip.structure.preparation import PDBComplex

top5 = res.head(5)
report = []

for _, r in top5.iterrows():
    lid = r["lig_id"]
    subprocess.run(["obabel", f"ligands/{lid}_out.pdbqt",
                    "-O", f"ligands/{lid}_pose.pdb",
                    "-f", "1", "-l", "1"],
                   capture_output=True)

    pose = [l for l in open(f"ligands/{lid}_pose.pdb").read().splitlines()
            if l.startswith(("ATOM", "HETATM"))]
    pose = [("HETATM" + l[6:17] + "LIG" + " " + "Z" + l[22:]) for l in pose]

    rec = [l for l in open("receptor.pdb").read().splitlines()
           if l.startswith(("ATOM", "HETATM"))]
    cpx = f"complex_{lid}.pdb"
    open(cpx, "w").write("\n".join(rec + pose) + "\nEND\n")

    try:
        m = PDBComplex()
        m.load_pdb(cpx)
        m.analyze()
        for key, site in m.interaction_sets.items():
            if "LIG" not in key:
                continue
            for hb in site.hbonds_pdon + site.hbonds_ldon:
                report.append(dict(lig=lid, score=r["score"], type="H-bond",
                                   residue=f"{hb.restype}{hb.resnr}",
                                   dist=round(hb.distance_ad, 2)))
            for hc in site.hydrophobic_contacts:
                report.append(dict(lig=lid, score=r["score"], type="Hydrophobic",
                                   residue=f"{hc.restype}{hc.resnr}",
                                   dist=round(hc.distance, 2)))
            for ps in site.pistacking:
                report.append(dict(lig=lid, score=r["score"], type="pi-stacking",
                                   residue=f"{ps.restype}{ps.resnr}",
                                   dist=round(ps.distance, 2)))
            for sb in site.saltbridge_lneg + site.saltbridge_pneg:
                report.append(dict(lig=lid, score=r["score"], type="Salt bridge",
                                   residue=f"{sb.restype}{sb.resnr}",
                                   dist=round(sb.distance, 2)))
    except Exception as e:
        print("PLIP failed for", lid, type(e).__name__, e)

inter = pd.DataFrame(report)
if len(inter):
    inter.to_csv("table4_interactions.csv", index=False)
    print(inter.to_string(index=False))
    print("\n\nInteraction counts per compound:")
    print(pd.crosstab(inter["lig"], inter["type"]).to_string())
    print("\nContacts involving FAD:")
    print(inter[inter["residue"].str.contains("FAD", na=False)].to_string(index=False))
else:
    print("No interactions parsed - inspect complex_*.pdb manually.")

## 6.10 Visualise the best pose

In [ ]:
import py3Dmol

best = res.iloc[0]["lig_id"]
view = py3Dmol.view(width=900, height=600)
view.addModel(open("receptor.pdb").read(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addStyle({"resn": "FAD"},
              {"stick": {"colorscheme": "yellowCarbon", "radius": 0.18}})
view.addModel(open(f"ligands/{best}_pose.pdb").read(), "pdb")
view.setStyle({"model": -1},
              {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})
view.zoomTo({"model": -1})
print(f"Best compound: {best}   score {res.iloc[0]['score']:.2f} kcal/mol")
print("yellow = FAD cofactor,  cyan = docked ligand")
view.show()

## 6.11 Final summary

In [ ]:
print("=" * 60)
print("LSD1 VIRTUAL SCREENING - SUMMARY")
print("=" * 60)
print(f"Target                : LSD1 / KDM1A (CHEMBL6136)")
print(f"Structure             : {PDB_ID}, FAD retained")
print(f"Curated compounds     : {len(df):,}")
print(f"Screening library     : {len(lib):,} (ZINC)")
print(f"Drug library          : {len(dl):,} (Broad Repurposing Hub)")
print(f"Similarity cutoff     : {CUTOFF}")
print(f"ZINC hits (clustered) : {len(zinc_hits)}")
print(f"Drug hits             : {len(drug_hits)}")
print(f"Ligands docked        : {len(res)}")
print(f"Actives in top 5%     : {pct_in_top5:.1f}%")

print("\nOutput files:")
for f in sorted(os.listdir(".")):
    if f.endswith((".csv", ".png")):
        print("   ", f)

---
## Interpretation checklist

1. **Did redocking pass?** If RMSD > 3 A, say so plainly and treat all
   rankings as unreliable.
2. **Is the enrichment factor above 1?** If actives are not ranked above
   inactives, docking has no discriminative power here - report that, and
   consider why (rigid receptor, no explicit waters, scoring-function limits).
3. **Do the top compounds contact FAD?** Contacts with the cofactor support
   binding in the catalytic site.
4. **Did any approved drug rank well?** Check its annotated mechanism in the
   Broad data - an unrelated mechanism scoring highly is more likely a
   docking artefact than a discovery.